## Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer

# Dataset

In [2]:
seasons = pd.read_csv("processed_data/feature_engineering_seasons.csv")
clusters = pd.read_csv("processed_data/municipalities_clusters.csv")
df = seasons.merge(clusters, left_on="municipality", right_on="name", how="left")
df.drop(columns=["name", "lat", "lon"], inplace=True)

In [3]:
date_cols = [
    "season_start",
    "season_end",
    "returns_end"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

df["season_duration"] = pd.to_timedelta(
    df["season_duration"]
).dt.days

df = df.sort_values(["municipality", "year"])

In [4]:
df["season_start_doy"] = (
    df["season_start"].dt.dayofyear
)

df["season_end_doy"] = (
    df["season_end"].dt.dayofyear
)

df["returns_end_doy"] = (
    df["returns_end"].dt.dayofyear
)

df["has_returns"] = (
    df["returns_num"] > 0
).astype(int)

In [5]:
geo_data = pd.read_csv("processed_data/municipalities.csv")

In [6]:
df = df.merge(geo_data, left_on="municipality", right_on="name", how="left")

In [7]:
df.drop(columns=["name", "loc_name", "place"], inplace=True)

# EM (Expanding Mean)

In [8]:
targets = [
    "season_start_doy",
    "season_end_doy",
    "season_duration",
    "season_cum_score"
]

# Historical data up to 2025
train_2026 = df[df["year"] <= 2025].copy()

# Generate forecasts for each municipality
forecast_results = []

for municipality in train_2026["municipality"].unique():

    history = train_2026[
        train_2026["municipality"] == municipality
    ].sort_values("year")

    predictions = {
        "year": 2026,
        "municipality": municipality
    }

    for target in targets:

        predictions[target] = history[target].mean()

    forecast_results.append(predictions)

forecast_2026_mean = pd.DataFrame(forecast_results)

display(forecast_2026_mean.head())

,year,municipality,season_start_doy,season_end_doy,season_duration,season_cum_score
0,2026,Albaredo per San Marco,108.666667,324.000000,215.333333,640.068992
1,2026,Albosaggia,127.166667,316.833333,189.666667,591.626517
2,2026,Andalo Valtellino,98.166667,325.666667,227.500000,796.056702
3,2026,Aprica,134.166667,306.833333,172.666667,802.071065
4,2026,Ardenno,127.333333,323.833333,196.500000,593.392546


# RF (Random Forest)

In [9]:
history_cols = [
    "season_start_doy",
    "season_end_doy",
    "season_duration",
    "season_cum_score"
]

# Feature lag 1
for col in history_cols:
    df[f"{col}_lag1"] = (
        df.groupby("municipality")[col]
        .shift(1)
    )

# Expanding historical means, excluding current year
for col in history_cols:
    df[f"{col}_mean_past"] = (
        df.groupby("municipality")[col]
        .transform(
            lambda x: x.shift(1).expanding().mean()
        )
    )

In [10]:
history_features = [
    "season_start_doy_lag1",
    "season_end_doy_lag1",
    "season_duration_lag1",
    "season_cum_score_lag1",
    "season_start_doy_mean_past",
    "season_end_doy_mean_past",
    "season_duration_mean_past",
    "season_cum_score_mean_past"
]

geo_features = ["ele", "lat", "lon"]

numeric_features = history_features + geo_features

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features)
    ]
)

In [12]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=400,
        max_depth=5,
        min_samples_leaf=3,
        min_samples_split=4,
        max_features=0.8,
        bootstrap=True,
        random_state=42,
        n_jobs=-1                   # uses all available processors (CPU cores). I could have manually selected 1, 2 etc 
    ))
])

In [13]:

# Prepare 2026 features

# One row per municipality, using 2025 as the reference
df_2026_features = (
    df[df["year"] == 2025]
    .copy()
    .reset_index(drop=True)
)

# Set forecasting year
df_2026_features["year"] = 2026

# Historical features: Lag 1
# The 2025 values become the lagged features for 2026
for col in history_cols:
    df_2026_features[f"{col}_lag1"] = df_2026_features[col]

# Historical expanding means up to 2025
for col in history_cols:

    means = (
        df[df["year"] <= 2025]
        .groupby("municipality")[col]
        .mean()
    )

    df_2026_features[f"{col}_mean_past"] = (
        df_2026_features["municipality"].map(means)
    )

# Check
display(
    df_2026_features[
        ["municipality", "year"] + numeric_features
    ].head()
)


,municipality,year,season_start_doy_lag1,season_end_doy_lag1,season_duration_lag1,season_cum_score_lag1,season_start_doy_mean_past,season_end_doy_mean_past,season_duration_mean_past,season_cum_score_mean_past,ele,lat,lon
0,Albaredo per San Marco,2026,99,357,258,648.305856,108.666667,324.000000,215.333333,640.068992,950,46.102733,9.590179
1,Albosaggia,2026,115,323,208,643.093334,127.166667,316.833333,189.666667,591.626517,490,46.147232,9.853170
2,Andalo Valtellino,2026,88,358,270,912.076338,98.166667,325.666667,227.500000,796.056702,229,46.132475,9.474441
3,Aprica,2026,119,318,199,842.760971,134.166667,306.833333,172.666667,802.071065,1181,46.154014,10.152581
4,Ardenno,2026,115,358,243,697.623116,127.333333,323.833333,196.500000,593.392546,266,46.171571,9.651944


In [14]:

# Training data: 2021-2025

train_2026 = df[
    (df["year"] >= 2021) &
    (df["year"] <= 2025)
].copy()

forecast_rf = []

for target in targets:

    rf_model.fit(
        train_2026[numeric_features],
        train_2026[target]
    )

    predictions = rf_model.predict(
        df_2026_features[numeric_features]
    )

    for municipality, pred in zip(
        df_2026_features["municipality"],
        predictions
    ):

        forecast_rf.append({
            "year": 2026,
            "municipality": municipality,
            "target": target,
            "prediction": pred
        })

forecast_rf = pd.DataFrame(forecast_rf)

# Convert to wide format
forecast_rf_wide = forecast_rf.pivot(
    index=["year", "municipality"],
    columns="target",
    values="prediction"
).reset_index()

display(forecast_rf_wide.head())

target,year,municipality,season_cum_score,season_duration,season_end_doy,season_start_doy
0,2026,Albaredo per San Marco,661.219612,199.480250,304.588223,107.372426
1,2026,Albosaggia,692.689016,196.233628,308.292986,110.670716
2,2026,Andalo Valtellino,790.683547,210.737198,306.041287,104.660374
3,2026,Aprica,831.116886,192.897422,305.807861,118.234840
4,2026,Ardenno,667.442489,185.919160,304.797762,122.055979


# Dataset 2026

In [15]:
# Merge Expanding Mean and Random Forest forecasts

forecast_2026_comparison = pd.merge(
    forecast_2026_mean,
    forecast_rf_wide,
    on=["year", "municipality"],
    how="inner",
    suffixes=("_mean", "_rf")
)

display(forecast_2026_comparison.head())

,year,municipality,season_start_doy_mean,season_end_doy_mean,season_duration_mean,season_cum_score_mean,season_cum_score_rf,season_duration_rf,season_end_doy_rf,season_start_doy_rf
0,2026,Albaredo per San Marco,108.666667,324.000000,215.333333,640.068992,661.219612,199.480250,304.588223,107.372426
1,2026,Albosaggia,127.166667,316.833333,189.666667,591.626517,692.689016,196.233628,308.292986,110.670716
2,2026,Andalo Valtellino,98.166667,325.666667,227.500000,796.056702,790.683547,210.737198,306.041287,104.660374
3,2026,Aprica,134.166667,306.833333,172.666667,802.071065,831.116886,192.897422,305.807861,118.234840
4,2026,Ardenno,127.333333,323.833333,196.500000,593.392546,667.442489,185.919160,304.797762,122.055979


In [17]:
forecast_2026_comparison.drop(columns=["season_start_doy_mean", "season_end_doy_rf", "season_duration_rf", "season_cum_score_mean"], inplace=True)

| Target           | Model  |
|------            |----    |
| season_cum_score | RF     |
| season_duration  | EM     |
| season_end_doy   | EM     |
| season_start_doy | RF     |

In [18]:
forecast_2026_comparison.head(2)

,year,municipality,season_end_doy_mean,season_duration_mean,season_cum_score_rf,season_start_doy_rf
0,2026,Albaredo per San Marco,324.000000,215.333333,661.219612,107.372426
1,2026,Albosaggia,316.833333,189.666667,692.689016,110.670716


In [19]:
forecast_2026_comparison["season_start"] = (
    pd.to_datetime("2026-01-01") +
    pd.to_timedelta(
        forecast_2026_comparison["season_start_doy_rf"].round() - 1,
        unit="D"
    )
)

forecast_2026_comparison["season_end"] = (
    pd.to_datetime("2026-01-01") +
    pd.to_timedelta(
        forecast_2026_comparison["season_end_doy_mean"].round() - 1,
        unit="D"
    )
)

In [20]:
forecast_2026_comparison.to_csv(
    "processed_data/forecast_2026.csv",
    index=False
)

# Checks

Actually, the model predicts separately season start, end and duration. However, ther are correlated

In [22]:
# Durata calcolata dalle date
forecast_2026_comparison["duration_from_dates"] = (
    forecast_2026_comparison["season_end"] -
    forecast_2026_comparison["season_start"]
).dt.days + 1

# Differenza tra durata prevista e durata ricavata dalle date
forecast_2026_comparison["duration_difference"] = (
    forecast_2026_comparison["season_duration_mean"] -
    forecast_2026_comparison["duration_from_dates"]
)

display(
    forecast_2026_comparison[
        [
            "municipality",
            "season_duration_mean",
            "duration_from_dates",
            "duration_difference"
        ]
    ].head(15)
)

,municipality,season_duration_mean,duration_from_dates,duration_difference
0,Albaredo per San Marco,215.333333,218,-2.666667
1,Albosaggia,189.666667,207,-17.333333
2,Andalo Valtellino,227.500000,222,5.500000
3,Aprica,172.666667,190,-17.333333
4,Ardenno,196.500000,203,-6.500000
5,Bema,216.166667,219,-2.833333
6,Berbenno di Valtellina,190.666667,208,-17.333333
7,Bianzone,174.166667,190,-15.833333
8,Bormio,117.000000,118,-1.000000
9,Buglio in Monte,196.333333,203,-6.666667
